In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler,OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
from underthesea import word_tokenize
from scipy.sparse import hstack

In [29]:
df=pd.read_csv("C:/Ky_2_nam_3/Recommend Product/crawler/data/merged_data.csv")
df=df.drop(columns=["categories","image","review"])
df.head()

,itemId,name,price,ratingScore,location,brandName,sellerName,itemSoldCntShow,discount,cat_1,cat_2,cat_3,category
0,3329615220,(Mp-TH) Hộp 60V Tăng Vòng 1 bổ sung nội tiết t...,245000,4.8,Vietnam,No Brand,Shop MP Thái - Việt Thiên Hoa,0.0,0.18,10100869,4741,6782,thuc_pham
1,2538405044,Bình đun siêu tốc 5 Lít JipLai ấm siêu tốc bền...,355000,4.7,Vietnam,Jiplai,Tổng kho điện máy 91,0.0,0.28,4400,4425,6037,dien_tu
2,3284439075,adidas | Slip-On Sports Sneakers,563000,4.8,China,adidas,JX.MAXX Outlets,0.0,0.00,62188201,1785,62540406,the_thao
3,2514651844,"Kem Giảm Thâm Mụn SANTAGIFT giúp trắng sáng,ức...",105000,4.8,Vietnam,No Brand,Her.Beautyhcm,688.0,0.30,4405,2277,2283,my_pham
4,3230429212,PUMA | Minimalist Stylish Low-top Canvas Shoes,1104000,5.0,China,PUMA,MY.JO sports shoes and clothing Specialty Store,8.0,0.00,42062401,7473,7501,the_thao


In [30]:


# Chuyển đổi các cột số sang định dạng đúng (đề phòng lỗi kiểu dữ liệu)
numeric_cols = ['price', 'ratingScore', 'itemSoldCntShow', 'discount']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Xử lý các cột ID danh mục thành dạng chuỗi để tránh tính toán sai lệch
cat_cols = ['cat_1', 'cat_2', 'cat_3']
df[cat_cols] = df[cat_cols].astype(str)

print("Dữ liệu đã sẵn sàng!")
df.head()

Dữ liệu đã sẵn sàng!


,itemId,name,price,ratingScore,location,brandName,sellerName,itemSoldCntShow,discount,cat_1,cat_2,cat_3,category
0,3329615220,(Mp-TH) Hộp 60V Tăng Vòng 1 bổ sung nội tiết t...,245000,4.8,Vietnam,No Brand,Shop MP Thái - Việt Thiên Hoa,0.0,0.18,10100869,4741,6782,thuc_pham
1,2538405044,Bình đun siêu tốc 5 Lít JipLai ấm siêu tốc bền...,355000,4.7,Vietnam,Jiplai,Tổng kho điện máy 91,0.0,0.28,4400,4425,6037,dien_tu
2,3284439075,adidas | Slip-On Sports Sneakers,563000,4.8,China,adidas,JX.MAXX Outlets,0.0,0.00,62188201,1785,62540406,the_thao
3,2514651844,"Kem Giảm Thâm Mụn SANTAGIFT giúp trắng sáng,ức...",105000,4.8,Vietnam,No Brand,Her.Beautyhcm,688.0,0.30,4405,2277,2283,my_pham
4,3230429212,PUMA | Minimalist Stylish Low-top Canvas Shoes,1104000,5.0,China,PUMA,MY.JO sports shoes and clothing Specialty Store,8.0,0.00,42062401,7473,7501,the_thao


In [31]:
def clean_product_name(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Loại bỏ nội dung trong ngoặc đơn và chính dấu ngoặc
    # \([^)]*\) khớp với dấu ( + bất kỳ ký tự nào không phải ) + dấu )
    text = re.sub(r'\([^)]*\)', '', text)
    
    # 2. Chuyển về chữ thường
    text = text.lower()
    
    # 3. Loại bỏ các ký tự đặc biệt còn lại (dấu : , . | - ...)
    # Để lại chữ cái, số và khoảng trắng
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # 4. Loại bỏ khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


In [32]:
def preprocess_vietnamese_text(text):
    # 1. Làm sạch (bỏ ngoặc, ký tự đặc biệt) như hàm bạn đã có
    cleaned = clean_product_name(text)
    
    # 2. Tách từ tiếng Việt
    tokenized = word_tokenize(cleaned, format="text")
    return tokenized

# Áp dụng cho toàn bộ dữ liệu
def calculate_cosine_sim(df):
    df['processed_name'] = df['name'].apply(preprocess_vietnamese_text)

# 3. Tính toán TF-IDF trên dữ liệu đã tách từ
    tfidf = TfidfVectorizer() 
    tfidf_matrix = tfidf.fit_transform(df['processed_name'])
    ohe = OneHotEncoder()
    cat_matrix = ohe.fit_transform(df[['cat_1', 'cat_2', 'cat_3']])


    combined_features = hstack([tfidf_matrix, cat_matrix])
    cosine_sim = cosine_similarity(combined_features,combined_features)
    return cosine_sim


In [ ]:
def get_recommendations_with_text(target_item_id, df, top_n=8):
    idx = df.index[df['itemId'] == target_item_id].tolist()[0]
    
    # Lấy điểm tương đồng văn bản từ ma trận
    sim_scores = list(enumerate(calculate_cosine_sim(df)[idx]))
    
    # Sắp xếp sản phẩm theo điểm tương đồng văn bản
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    item_indices = [i[0] for i in sim_scores]
    return df.iloc[item_indices]

In [34]:
def recommend_products(df):
    model=MinMaxScaler()
    df[["ratingScore","itemSoldCntShow","discount","price"]]=model.fit_transform(df[["ratingScore","itemSoldCntShow","discount","price"]])
    df["price"]=1-df["price"]
    df["score"]=df["ratingScore"]*0.4+df["itemSoldCntShow"]*0.3+df["discount"]*0.2+df["price"]*0.1
    return df.sort_values(by=['ratingScore'], ascending=False)

In [35]:
test=recommend_products(get_recommendations_with_text(2538405044,df))
test

C:\Users\Thanh Long\AppData\Local\Temp\ipykernel_9984\4025340362.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[["ratingScore","itemSoldCntShow","discount","price"]]=model.fit_transform(df[["ratingScore","itemSoldCntShow","discount","price"]])
C:\Users\Thanh Long\AppData\Local\Temp\ipykernel_9984\4025340362.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["price"]=1-df["price"]
C:\Users\Thanh Long\AppData\Local\Temp\ipykernel_9984\4025340362.py:5: SettingWithCopyWarning: 
A value is trying to 

,itemId,name,price,ratingScore,location,brandName,sellerName,itemSoldCntShow,discount,cat_1,cat_2,cat_3,category,processed_name,score
665,422906645,"Ấm siêu tốc, Bình Đun Nước Siêu Tốc 1,8L inox,...",0.978261,1.00,Vietnam,No Brand,GIADUNG-DALINHVUC,0.000000,0.531915,4400,4425,6037,dien_tu,ấm siêu_tốc bình đun nước siêu_tốc 1 8 l inox ...,0.604209
1650,2074748032,Ấm đun nước siêu tốc 1.8L Electric Kettle - ấm...,1.000000,1.00,Vietnam,No Brand,Kim Khí _Tổng Hợp,0.000000,0.000000,4400,4425,6037,dien_tu,ấm đun nước siêu_tốc 1 8 l electric kettle ấm ...,0.500000
1059,2650344292,Ấm siêu tốc Electric Kettle Inox 1.8L - Bình đ...,1.000000,1.00,Vietnam,No Brand,Bon an gia dụng,0.005882,0.936170,4400,4425,6037,dien_tu,ấm siêu_tốc electric kettle inox 1 8 l bình_đu...,0.688999
399,2013117646,"Ấm đun siêu tốc, Bình đun nước siêu tốc 2.3L -...",0.543478,0.50,Vietnam,ZODAN,ZODAN Corporation,0.764706,0.957447,4400,4425,6037,do_gia_dung,ấm đun siêu_tốc bình đun nước siêu_tốc 2 3 l b...,0.675249
463,2075174031,"[HÀNG TỐT] Ấm đun siêu tốc, Bình đun nước siêu...",0.565217,0.50,Vietnam,ZODAN,CÔNG TY HÀNG SỈ RUBI,1.000000,1.000000,4400,4425,6037,dien_tu,hàng tốt ấm đun siêu_tốc bình đun nước siêu_tố...,0.756522
774,2201365216,Bảo Hành 6 tháng - Ấm đun nước siêu tốc Thái L...,1.000000,0.25,Vietnam,No Brand,DOTI Plus,0.475882,0.765957,4400,4425,6037,do_gia_dung,bảo_hành 6 tháng ấm đun nước siêu_tốc thái_lan...,0.495956
1103,2609973143,Ấm Đun Siêu Tốc Jiplai 2.5L,0.000000,0.25,Vietnam,No Brand,Phụ Kiện Số Gia Hân,0.000000,0.000000,4400,4425,6037,dien_tu,ấm đun siêu_tốc jiplai 2 5 l,0.100000
239,1446585298,ẤM SIÊU TỐC INOX,0.869565,0.00,Vietnam,LS,Hoàn Cầu Electric,0.020000,0.000000,4400,4425,6037,do_gia_dung,ấm siêu_tốc inox,0.092957
